# 04. Sets

**Tier:** Intermediate. **Why you'd use this:** Two guest lists overlap. You want each name once, a fast "is this person invited?" check, and the names that appear on both lists or only on one.

**After this lesson you can:** keep each value once; test membership without scanning a list; choose `&` or `-` for overlap and difference.
About 25 minutes. **Uses:** [Dicts](03-dicts-lesson.ipynb)


## Quick review

1. Why is `"to" in counts` usually faster than `"to" in words` when `counts` is a dict?
2. What does `counts.get("missing", 0)` return when the key is absent?

<details><summary>Answers</summary>

Dict lookup jumps by hash instead of scanning. `0` — the default.
</details>


## The idea

A set keeps each value once. `in` asks whether it is there, by the same hashing a dict uses, so it does not walk every item. `&` is the overlap. `-` is what the left set has that the right set does not. A set has no index, and you cannot rely on the order it prints.

<details><summary>Explain more</summary>

`set(nums)` drops duplicates. Adding a value that is already there changes nothing. Items must be hashable, same rule as dict keys. A list cannot be a set item.
</details>


## See it work

Drop repeated names, and keep the first time each name appeared.


In [1]:
def first_seen(names):
    # 1. The set answers "already recorded?" in one jump.
    seen = set()
    kept = []
    # 2. Append only the first time. The list remembers the order.
    for name in names:
        if name not in seen:
            seen.add(name)
            kept.append(name)
    return kept


print(first_seen(["bea", "ada", "bea", "cy"]))

['bea', 'ada', 'cy']


## Check yourself

Answer these from the example above before you run the cell.


1. Why does `first_seen` keep a list as well as a set?
2. What is `{1} & {2}`?

<details><summary>Answers</summary>

The set answers membership and forgets arrival order. The list stores that order. Two sets with nothing in common overlap in the empty set.
</details>


In [2]:
print(list({3, 1, 2}))
print(2 in {1, 2})
print({1, 2, 3} & {2, 3, 4})
print({1, 2} - {2, 3})

[1, 2, 3]
True
{2, 3}
{1}


<details><summary>Check your answers</summary>

The printed order is not `3`, then `1`, then `2`. A set does not promise insertion order, so don't use that order in your program. `2 in {1, 2}` is true. `&` keeps `2` and `3`. `-` keeps `1`.
</details>


## Why it works this way

A set is a hash table of keys with no value beside them. `add` hashes the item and stores it in a slot. `in` hashes again and checks that slot. It does not compare against every member, so membership is average O(1), the same reason a dict lookup is. The memory is the table, including empty slots that keep collisions rare. A list's `in` is an O(n) scan. The timed cell is that difference on a large collection.

Items must be hashable, for the same reason dict keys must. If the item changed, the slot would be wrong and `in` would miss it. A list is rejected with `unhashable type: 'list'`. Empty slots are the memory you pay so a probe stays short. If the table were packed solid, new items would collide and `in` would slide toward a scan. A removed item cannot leave a hole that stops the probe early, or a later item that hashed past that hole would look missing.

Order is not part of the set's contract. The print of `{3, 1, 2}` is not the order the values were written. Dicts promise insertion order. Sets do not, so a set cannot number lines or remember who arrived first. If you need that order, keep a list as well, the way `first_seen` does. `&` and `-` each walk one set and probe the other, so they cost about the size of the sets involved, not the product. Deep dive: [How dict and set work](../../internals/04-dicts/01-hash-table-lesson.ipynb).


In [3]:
import timeit

print(
    "list",
    timeit.timeit(
        "99999 in nums",
        setup="nums = list(range(100000))",
        number=200,
    ),
)
print(
    "set",
    timeit.timeit(
        "99999 in found",
        setup="found = set(range(100000))",
        number=200,
    ),
)

list 0.138301875005709
set 6.374990334734321e-06


## When to use it

Use a set for membership, duplicates, overlap, and difference. Use a dict when each key needs a value. Use a list when order is the data. Don't use a set as a queue or as line numbers. Don't use a list's `in` inside a loop over a large collection when a set would answer the same question.


## Common mistake

Putting a list into a set, often a row or a pair of coordinates stored as a list. The set cannot hash it, because the list could change. The last line is `TypeError`. The arrow marks the `set` call.


In [4]:
def track(rows):
    return set(rows)


print(track([[1, 2], [3, 4]]))

TypeError: cannot use 'list' as a set element (unhashable type: 'list')

A tuple of numbers can be hashed. The coordinates stay fixed, so the set can store each point once.


In [5]:
def track(rows):
    points = set()
    for row in rows:
        points.add((row[0], row[1]))
    return points


print(sorted(track([[1, 2], [3, 4], [1, 2]])))

[(1, 2), (3, 4)]


## Try it

Change `first_seen` so a repeated name is kept at its last position, not its first. What do you remove from the list before you append again, and why is the set alone not enough?


## Recap

- A set hashes items. Membership is average O(1). Duplicates disappear.
- `&` is the overlap. `-` is the difference. There is no index.
- Don't rely on print order. A list beside the set remembers order.

Next: [practice](04-sets-practice.ipynb).
